# 因子优化与IC增强工程

## 基于量化金融文献的因子优化方法集成

> 参考文献：
> - Kakushadze (2016) "101 Formulaic Alphas" — Rank/Z-Score/Decay/SignedPower 算子体系
> - Barroso & Santa-Clara (2015) RFS — 波动率缩放动量
> - 中金公司 (2025) "Alpha提纯" — 正交残差化
> - WorldQuant BRAIN — 因子精炼管线
> - 信达证券 (2024) "深度学习因子中性化"
> - Tulchinsky (2019) "Finding Alphas" (2nd Ed.)

## 优化方法概览

| 类别 | 方法 | 文献支撑 | 预期效果 |
|------|------|----------|----------|
| **A. 时序排序 Rank** | 滚动窗口内排序映射至[0,1] | WorldQuant 101 Alphas | 消除异常值、分布均匀化 |
| **B. 滚动Z-Score** | $(F_t - \mu_w)/\sigma_w$ | 业界标准 | 相对位置归一化、跨时段可比 |
| **C. 波动率缩放** | $F_t/\sigma_{GK,w}$ | Barroso & Santa-Clara (2015) | 风险调整、提升IR稳定性 |
| **D. 衰减线性加权** | 近期数据权重更大 | WorldQuant decay_linear | 降低滞后、提升响应速度 |
| **E. 正交残差化** | 回归剥离共同风险 | 中金Alpha提纯 (2025) | 纯Alpha信号、降低共线性 |
| **F. 因子动量 Delta** | $F_t - F_{t-d}$ | 业界实践 | 捕捉因子变化速度 |
| **G. 有符号幂变换** | $\text{sign}(x) \cdot |x|^p$ | WorldQuant SignedPower | 非线性放大、保留方向 |
| **H. 交叉共振** | $F_1 \times F_2$ | 量价背离文献 | 捕捉因子间非线性交互 |
| **I. 多周期差分** | 短周期-长周期对比 | 业界实践 | 捕捉动量加速/衰竭 |

---

共设计 **25个优化因子**，覆盖9大优化方法类别。

In [ ]:
import os, time, gc
import numpy as np
import pandas as pd
import talib
import bottleneck as bn
from numpy.lib.stride_tricks import sliding_window_view
from pathlib import Path
from numpy.lib.stride_tricks import sliding_window_view

BASE_DIR = Path(r"C:\Users\wangx\Desktop\华东")
RAW_DATA_DIR = BASE_DIR / "train_dataset" / "train_dataset"
OUTPUT_CSV = BASE_DIR / "Optimized_Factors_IC_Evaluation.csv"
DATASET_COUNT = 30

# 25个优化因子
FACTOR_NAMES = [
    # === A. 时序排序 Rank (4) ===
    "A1_TsRank5_GK_Vol",            # GK波动率的5期时序排名
    "A2_TsRank20_OrderImbalance",   # 订单失衡的20期时序排名
    "A3_TsRank60_LogHL_Range",      # 对数极差的60期时序排名
    "A4_TsRank10_VWAP_Dev",         # VWAP偏离的10期时序排名
    # === B. 滚动Z-Score (4) ===
    "B1_ZScore60_GK_Vol",           # GK波动率的60期Z-Score
    "B2_ZScore60_OrderImbalance",   # 订单失衡的60期Z-Score
    "B3_ZScore60_LogHL_Range",      # 对数极差的60期Z-Score
    "B4_ZScore30_Close_Position",   # 收盘位置的30期Z-Score
    # === C. 波动率缩放 (4) ===
    "C1_VolScaled_ROC_5M",          # ROC_5M / GK波动率(5期)
    "C2_VolScaled_ROC_60M",         # ROC_60M / GK波动率(60期)
    "C3_VolScaled_MACD_60M",        # MACD / GK波动率(60期)
    "C4_VolScaled_OrderImbalance",  # 订单失衡 / GK波动率(20期)
    # === D. 衰减线性加权 (3) ===
    "D1_DecayLinear5_Returns",      # 5期线性衰减加权收益率
    "D2_DecayLinear10_GK_Vol",      # 10期线性衰减加权GK波动率
    "D3_DecayLinear20_LogHL",       # 20期线性衰减加权对数极差
    # === E. 正交残差化 (3) ===
    "E1_Orth_EMA_Ret_vs_GKVol",     # EMA乖离对GK波动率正交
    "E2_Orth_VWAP_Dev_vs_LogHL",    # VWAP偏离对LogHL正交
    "E3_Orth_OrderImb_vs_GKVol",    # 订单失衡对GK波动率正交
    # === F. 因子动量 Delta (3) ===
    "F1_Delta3_OrderImbalance",     # 订单失衡的3期变化
    "F2_Delta5_GK_Vol",             # GK波动率的5期变化
    "F3_Delta5_LogHL_Range",        # LogHL的5期变化
    # === G. 有符号幂变换 (2) ===
    "G1_SignedPow2_LogRet",         # LogRet的有符号平方
    "G2_SignedPow15_VWAP_Dev",      # VWAP_Dev的有符号1.5次方
    # === H. 交叉共振 (2) ===
    "H1_OI_x_GKVol",                # 订单失衡 × GK波动率
    "H2_VWAP_Dev_x_LogHL",          # VWAP偏离 × LogHL
]
FACTOR_DIM = len(FACTOR_NAMES)
print(f"共 {FACTOR_DIM} 个优化因子")


In [ ]:
# ==========================================
# 基础计算函数
# ==========================================

def compute_base_factors(O, H, L, C, V):
    """计算需要的基础因子"""
    eps = 1e-8
    C_64, H_64, L_64, V_64 = C.astype(np.float64), H.astype(np.float64), L.astype(np.float64), V.astype(np.float64)
    O_64 = O.astype(np.float64)

    # Level 1 基础
    log_hl = np.log(np.maximum(H_64, L_64 + eps) / np.maximum(L_64, eps))
    log_oc = np.log(np.maximum(O_64, C_64 + eps) / np.maximum(C_64, eps))
    gk_vol = np.maximum(0.5 * log_hl**2 - (2*np.log(2)-1) * log_oc**2, 0)
    log_hl_range = log_hl
    min_oc = np.minimum(O_64, C_64)
    hl_range = H_64 - L_64 + eps
    close_position = (C_64 - L_64) / hl_range
    log_ret = np.log(np.maximum(C_64, eps)) - np.log(np.maximum(np.roll(C_64, 1), eps))
    log_ret[0] = 0.0

    # Order Imbalance
    delta_c = C_64 - np.roll(C_64, 1)
    delta_c[0] = 0.0
    order_imbalance = delta_c / (np.log(np.maximum(V_64, 1)) + eps)

    # VWAP deviation (single bar)
    ohlc4 = (O_64 + H_64 + L_64 + C_64) / 4.0
    vwap_dev = (C_64 - ohlc4) / (ohlc4 + eps)

    # ROC 5M and 60M
    roc_5 = (C_64 - np.roll(C_64, 5)) / (np.roll(C_64, 5) + eps)
    roc_5[:5] = 0.0
    roc_60 = (C_64 - np.roll(C_64, 60)) / (np.roll(C_64, 60) + eps)
    roc_60[:60] = 0.0

    # MACD
    ema12 = talib.EMA(C_64, timeperiod=12)
    ema26 = talib.EMA(C_64, timeperiod=26)
    ema12 = np.nan_to_num(ema12, nan=C_64[0])
    ema26 = np.nan_to_num(ema26, nan=C_64[0])
    macd_line = ema12 - ema26

    # EMA乖离
    ema60 = talib.EMA(C_64, timeperiod=60)
    ema60 = np.nan_to_num(ema60, nan=C_64[0])
    ema_ret = (C_64 - ema60) / (ema60 + eps)

    return {
        'gk_vol': gk_vol.astype(np.float32),
        'log_hl_range': log_hl_range.astype(np.float32),
        'close_position': close_position.astype(np.float32),
        'log_ret': log_ret.astype(np.float32),
        'order_imbalance': order_imbalance.astype(np.float32),
        'vwap_dev': vwap_dev.astype(np.float32),
        'roc_5': roc_5.astype(np.float32),
        'roc_60': roc_60.astype(np.float32),
        'macd_line': macd_line.astype(np.float32),
        'ema_ret': ema_ret.astype(np.float32),
        'hl_range': hl_range.astype(np.float32),
    }


def ts_rank(x, window):
    """时序排名: 向量化版本, 使用sliding_window_view加速"""
    T = len(x)
    result = np.full(T, 0.5, dtype=np.float32)
    if T <= window:
        return result
    sw = sliding_window_view(x.astype(np.float64), window)
    last_col = sw[:, -1:]  # (N, 1)
    result[window-1:] = (sw <= last_col).mean(axis=1)
    return result


def rolling_zscore(x, window):
    """滚动Z-Score: (x - mean_w) / std_w"""
    eps = 1e-8
    mean_w = bn.move_mean(x, window=window, min_count=1)
    std_w = bn.move_std(x, window=window, min_count=1)
    result = (x - mean_w) / (std_w + eps)
    return np.nan_to_num(result, nan=0.0).astype(np.float32)


def rolling_beta(x, y, window):
    """滚动Beta: Cov(x,y) / Var(y) over window"""
    eps = 1e-8
    mean_x = bn.move_mean(x, window=window, min_count=1)
    mean_y = bn.move_mean(y, window=window, min_count=1)
    cov_xy = bn.move_mean(x * y, window=window, min_count=1) - mean_x * mean_y
    var_y = bn.move_var(y, window=window, min_count=1)
    return cov_xy / (var_y + eps)


def decay_linear_weights(window):
    """线性衰减权重: w, w-1, ..., 1 归一化"""
    weights = np.arange(window, 0, -1, dtype=np.float64)
    return weights / weights.sum()


def rolling_decay_linear(x, window):
    """滚动衰减线性加权平均"""
    T = len(x)
    result = np.full(T, np.nan, dtype=np.float64)
    w = decay_linear_weights(window)
    for t in range(window - 1, T):
        result[t] = np.dot(x[t - window + 1 : t + 1], w)
    result[:window-1] = bn.move_mean(x, window=window, min_count=1)[:window-1]
    return np.nan_to_num(result, nan=0.0).astype(np.float32)


In [ ]:
# ==========================================
# 优化因子生成主函数
# ==========================================

def generate_optimized_factors(O, H, L, C, V):
    """生成全部25个优化因子"""
    bf = compute_base_factors(O, H, L, C, V)
    factors = []

    # ===== A. 时序排序 Rank (4) =====
    factors.append(ts_rank(bf['gk_vol'], 5))           # A1
    factors.append(ts_rank(bf['order_imbalance'], 20))  # A2
    factors.append(ts_rank(bf['log_hl_range'], 60))     # A3
    factors.append(ts_rank(bf['vwap_dev'], 10))         # A4

    # ===== B. 滚动Z-Score (4) =====
    factors.append(rolling_zscore(bf['gk_vol'], 60))         # B1
    factors.append(rolling_zscore(bf['order_imbalance'], 60)) # B2
    factors.append(rolling_zscore(bf['log_hl_range'], 60))    # B3
    factors.append(rolling_zscore(bf['close_position'], 30))  # B4

    # ===== C. 波动率缩放 (4) =====
    eps = 1e-8
    gk_vol_5 = np.sqrt(np.maximum(bn.move_mean(bf['gk_vol'], window=5, min_count=1), eps))
    gk_vol_20 = np.sqrt(np.maximum(bn.move_mean(bf['gk_vol'], window=20, min_count=1), eps))
    gk_vol_60 = np.sqrt(np.maximum(bn.move_mean(bf['gk_vol'], window=60, min_count=1), eps))

    factors.append((bf['roc_5'] / (gk_vol_5 + eps)).astype(np.float32))        # C1
    factors.append((bf['roc_60'] / (gk_vol_60 + eps)).astype(np.float32))       # C2
    factors.append((bf['macd_line'] / (gk_vol_60 + eps)).astype(np.float32))    # C3
    factors.append((bf['order_imbalance'] / (gk_vol_20 + eps)).astype(np.float32)) # C4

    # ===== D. 衰减线性加权 (3) =====
    factors.append(rolling_decay_linear(bf['log_ret'], 5))      # D1
    factors.append(rolling_decay_linear(bf['gk_vol'], 10))      # D2
    factors.append(rolling_decay_linear(bf['log_hl_range'], 20)) # D3

    # ===== E. 正交残差化 (3) =====
    # E1: Orth_EMA_Ret vs GK_Vol
    beta_ema_gk = rolling_beta(bf['ema_ret'], bf['gk_vol'], 60)
    orth_ema = bf['ema_ret'] - beta_ema_gk * bf['gk_vol']
    factors.append(orth_ema.astype(np.float32))

    # E2: Orth_VWAP_Dev vs LogHL
    beta_vwap_hl = rolling_beta(bf['vwap_dev'], bf['log_hl_range'], 60)
    orth_vwap = bf['vwap_dev'] - beta_vwap_hl * bf['log_hl_range']
    factors.append(orth_vwap.astype(np.float32))

    # E3: Orth_OrderImb vs GK_Vol
    beta_oi_gk = rolling_beta(bf['order_imbalance'], bf['gk_vol'], 60)
    orth_oi = bf['order_imbalance'] - beta_oi_gk * bf['gk_vol']
    factors.append(orth_oi.astype(np.float32))

    # ===== F. 因子动量 Delta (3) =====
    factors.append((bf['order_imbalance'] - np.roll(bf['order_imbalance'], 3)).astype(np.float32))  # F1
    factors.append((bf['gk_vol'] - np.roll(bf['gk_vol'], 5)).astype(np.float32))                     # F2
    factors.append((bf['log_hl_range'] - np.roll(bf['log_hl_range'], 5)).astype(np.float32))           # F3

    # ===== G. 有符号幂变换 (2) =====
    logret = bf['log_ret'].astype(np.float64)
    signed_pow2 = np.sign(logret) * (np.abs(logret) ** 2.0)
    factors.append(signed_pow2.astype(np.float32))  # G1

    vwap_dev = bf['vwap_dev'].astype(np.float64)
    signed_pow15 = np.sign(vwap_dev) * (np.abs(vwap_dev) ** 1.5)
    factors.append(signed_pow15.astype(np.float32))  # G2

    # ===== H. 交叉共振 (2) =====
    factors.append((bf['order_imbalance'] * bf['gk_vol']).astype(np.float32))  # H1
    factors.append((bf['vwap_dev'] * bf['log_hl_range']).astype(np.float32))  # H2

    return np.column_stack(factors).astype(np.float32)


In [ ]:
# ==========================================
# IC 计算引擎
# ==========================================

def calculate_pearson_ic(factors, label):
    """计算每个因子与label的Pearson IC"""
    valid_mask = ~np.isnan(label)
    y = label[valid_mask]
    X = factors[valid_mask, :]

    X_nan_mask = np.isnan(X)
    X = np.where(X_nan_mask, 0, X)

    valid_count = (~X_nan_mask).sum(axis=0)
    valid_count = np.where(valid_count == 0, 1e-8, valid_count)

    y_mean = np.mean(y)
    X_mean = X.sum(axis=0) / valid_count

    y_centered = y - y_mean
    X_centered = np.where(X_nan_mask, 0, X - X_mean)

    cov = np.dot(y_centered, X_centered) / valid_count
    y_std = np.sqrt(np.mean(y_centered**2))
    X_std = np.sqrt(np.sum(X_centered**2, axis=0) / valid_count)

    ic = cov / (y_std * X_std + 1e-8)
    ic = np.where(valid_count < len(y) * 0.1, 0, ic)
    return ic


In [ ]:
# ==========================================
# 主流水线: 读取 -> 生成优化因子 -> IC评测
# ==========================================

def evaluate_optimized_factors():
    print("=" * 75)
    print("[因子优化引擎] 基于文献方法的25个优化因子 IC评估")
    print("=" * 75)
    print(f"\n优化方法: Rank | Z-Score | Vol-Scaling | Decay | Orthogonalization |")
    print(f"          Delta | SignedPower | Cross-Interaction")
    print(f"因子数量: {FACTOR_DIM}")
    print()

    ic_ret5 = np.zeros((DATASET_COUNT, FACTOR_DIM), dtype=np.float32)
    ic_ret60 = np.zeros((DATASET_COUNT, FACTOR_DIM), dtype=np.float32)

    success = 0
    t0 = time.time()

    for i in range(DATASET_COUNT):
        ds_name = f"dataset{i}"
        raw_path = RAW_DATA_DIR / f"{ds_name}_train_ohlcv.npy"

        if not raw_path.exists():
            continue

        try:
            raw = np.load(raw_path).astype(np.float32)
            if raw.shape[1] != 8:
                continue

            O = raw[:, 1]
            H = raw[:, 2]
            L = raw[:, 3]
            C = raw[:, 4]
            V = raw[:, 5]
            ret5 = raw[:, 6]
            ret60 = raw[:, 7]

            # 生成25个优化因子
            opt_factors = generate_optimized_factors(O, H, L, C, V)

            # IC计算
            ic_ret5[i, :] = calculate_pearson_ic(opt_factors, ret5)
            ic_ret60[i, :] = calculate_pearson_ic(opt_factors, ret60)

            success += 1
            if success % 5 == 0:
                print(f"  已完成 {success}/{DATASET_COUNT} 个数据集...")

            del raw, opt_factors
            gc.collect()

        except Exception as e:
            print(f"  {ds_name} 失败: {str(e)[:60]}")

    elapsed = (time.time() - t0) / 60
    print(f"\n[完成] {success}/{DATASET_COUNT} 个数据集, 耗时 {elapsed:.2f} 分钟")

    # 汇总统计
    vic5 = ic_ret5[:success, :]
    vic60 = ic_ret60[:success, :]

    mean5 = np.nanmean(vic5, axis=0)
    mean60 = np.nanmean(vic60, axis=0)
    std5 = np.nanstd(vic5, axis=0)
    std60 = np.nanstd(vic60, axis=0)
    ir5 = mean5 / (std5 + 1e-8)
    ir60 = mean60 / (std60 + 1e-8)
    score5 = np.abs(mean5) * np.abs(ir5)
    score60 = np.abs(mean60) * np.abs(ir60)

    # 优化方法标签
    method_map = {
        'A1': 'A.时序排序', 'A2': 'A.时序排序', 'A3': 'A.时序排序', 'A4': 'A.时序排序',
        'B1': 'B.滚动Z-Score', 'B2': 'B.滚动Z-Score', 'B3': 'B.滚动Z-Score', 'B4': 'B.滚动Z-Score',
        'C1': 'C.波动率缩放', 'C2': 'C.波动率缩放', 'C3': 'C.波动率缩放', 'C4': 'C.波动率缩放',
        'D1': 'D.衰减加权', 'D2': 'D.衰减加权', 'D3': 'D.衰减加权',
        'E1': 'E.正交残差', 'E2': 'E.正交残差', 'E3': 'E.正交残差',
        'F1': 'F.因子动量Delta', 'F2': 'F.因子动量Delta', 'F3': 'F.因子动量Delta',
        'G1': 'G.有符号幂', 'G2': 'G.有符号幂',
        'H1': 'H.交叉共振', 'H2': 'H.交叉共振',
    }
    methods = [method_map.get(n.split('_')[0], 'Unknown') for n in FACTOR_NAMES]

    # 文献
    lit_map = {
        'A': 'WorldQuant (2016) ts_rank算子',
        'B': '业界标准 Z-Score归一化',
        'C': 'Barroso & Santa-Clara (2015) RFS',
        'D': 'WorldQuant decay_linear算子',
        'E': '中金Alpha提纯 (2025)',
        'F': '因子动量 业界实践',
        'G': 'WorldQuant SignedPower算子',
        'H': '量价背离文献',
    }
    lits = [lit_map.get(n.split('_')[0][0], '') for n in FACTOR_NAMES]

    df = pd.DataFrame({
        'Factor_Name': FACTOR_NAMES,
        'Method': methods,
        'Literature_Ref': lits,
        'Ret5_Mean_IC': mean5, 'Ret5_Std_IC': std5, 'Ret5_IC_IR': ir5,
        'Ret60_Mean_IC': mean60, 'Ret60_Std_IC': std60, 'Ret60_IC_IR': ir60,
        'Ret5_Score': score5, 'Ret60_Score': score60,
        'Comprehensive_Score': score5 + score60
    })

    # 打印结果
    print("\n" + "=" * 75)
    print("[Ret5 短线] Top 10 优化因子:")
    print("=" * 75)
    top5_5 = df.sort_values('Ret5_Score', ascending=False).head(10)
    print(top5_5[['Factor_Name', 'Method', 'Ret5_Mean_IC', 'Ret5_IC_IR', 'Ret5_Score']].to_string(index=False))

    print("\n" + "=" * 75)
    print("[Ret60 长线] Top 10 优化因子:")
    print("=" * 75)
    top5_60 = df.sort_values('Ret60_Score', ascending=False).head(10)
    print(top5_60[['Factor_Name', 'Method', 'Ret60_Mean_IC', 'Ret60_IC_IR', 'Ret60_Score']].to_string(index=False))

    print("\n" + "=" * 75)
    print("[综合排名] 全部优化因子:")
    print("=" * 75)
    df_sorted = df.sort_values('Comprehensive_Score', ascending=False)
    print(df_sorted[['Factor_Name', 'Method', 'Ret5_Mean_IC', 'Ret5_IC_IR',
                      'Ret60_Mean_IC', 'Ret60_IC_IR', 'Comprehensive_Score']].to_string(index=False))

    # 按方法类别汇总
    print("\n" + "=" * 75)
    print("[方法类别汇总] 各类方法平均综合得分:")
    print("=" * 75)
    for m in sorted(df['Method'].unique()):
        sub = df[df['Method'] == m]
        avg_score = sub['Comprehensive_Score'].mean()
        max_score = sub['Comprehensive_Score'].max()
        best_name = sub.loc[sub['Comprehensive_Score'].idxmax(), 'Factor_Name']
        print(f"  {m:25s}  平均={avg_score:.6f}  最高={max_score:.6f} ({best_name})")

    # 保存
    df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    print(f"\n[保存] 结果已输出至: {OUTPUT_CSV}")
    print("=" * 75)

    return df

if __name__ == "__main__":
    df_result = evaluate_optimized_factors()


# Signed Power 有符号幂变换 — 对现有因子库的优化

## 方法原理

基于上一轮优化的实证发现，**有符号幂变换 (Signed Power)** 是综合得分最高的优化方法（平均得分 0.0646）。

### 数学定义
$$\text{SignedPower}(x, p) = \text{sign}(x) \cdot |x|^p$$

### 优化逻辑

| 因子类型 | 推荐幂次 | 效果 |
|----------|:------:|------|
| **有符号振荡型** (VWAP偏离、动量、MACD、RSI中心化、PPO等) | p=1.5 | 温和放大极端信号，保留方向 |
| **有符号振荡型** (收益率、订单失衡) | p=2.0 | 强力放大极端信号 |
| **正值重尾型** (波动率、极差、Amihud) | p=0.5 | 压缩极端值（平方根），缩小分布尾部 |
| **正值重尾型** (极度重尾) | p=0.3 | 强力压缩 |

### 筛选策略
从现有311维因子库中选取 **IC排名靠前且分布适合幂变换** 的因子进行优化：
- 有符号因子（可正可负）：应用 p=1.5 和 p=2.0
- 正值因子（始终>=0）：应用 p=0.3 和 p=0.5

共生成 **35个Signed Power优化因子**。

## Signed Power 有符号幂变换 — 方法论详解

### 1. 数学定义

$$\text{SignedPower}(x, p) = \text{sign}(x) \cdot |x|^p$$

其中 $x$ 为原始因子值，$p$ 为幂次参数。该变换的核心特性：
- **保留方向**：$\text{sign}(x)$ 确保变换后的符号与原始值一致
- **非线性缩放**：$|x|^p$ 根据 $p$ 的取值实现放大或压缩

### 2. 幂次选择策略

| 幂次 $p$ | 变换名称 | 数学效果 | 适用场景 |
|:------:|:--------|:--------|:--------|
| $p=1.5$ | **温和放大** | 中等值线性放大，极端值非线性增强 | 有符号振荡型因子（VWAP偏离、动量、MACD等） |
| $p=2.0$ | **强力放大** | 小信号基本不变，大信号平方增强 | IC较高的有符号因子（收益率、强趋势信号） |
| $p=0.5$ | **平方根压缩** | 压缩极端值，缩小尾部 | 正值重尾型因子（波动率、极差、ATR） |
| $p=0.3$ | **强力压缩** | 极端值大幅压缩 | 极度重尾正值因子（跳跃比率、Amihud非流动性） |

### 3. 因子类型与幂次匹配

#### 有符号振荡型 $\rightarrow$ $p=1.5$ (SP15) 或 $p=2.0$ (SP20)
这类因子可正可负，中心在0附近。应用有符号幂变换后：
- 保留原始信号方向（做多/做空）
- 放大极端信号（高确信度信号权重更大）
- 压缩微弱噪声（中心附近信号被抑制）

**适用因子举例**：VWAP偏离、订单失衡、EMA乖离、ROC、MACD、PPO、TRIX、CCI、RSI中心化、CMF、Williams %R、CMO、Aroon振荡器等。

#### 正值重尾型 $\rightarrow$ $p=0.5$ (SP05) 或 $p=0.3$ (SP03)
这类因子始终非负（波动率、极差、非流动性），分布重尾。应用幂压缩后：
- 缩小极端值的影响
- 使分布更接近正态
- 提升IC稳定性

**适用因子举例**：GK波动率、对数极差、归一化ATR、ADX、实现波动率、跳跃比率、Amihud非流动性等。

#### 正值有界型 $[0,1] \rightarrow p=2.0$ (SP20)
这类因子在 $[0,1]$ 区间内（收盘位置、实体占比），应用平方后：
- 放大接近1的极端信号
- 抑制接近0的微弱信号

### 4. 文献支撑

- **WorldQuant (Zura Kakushadze, 2016)**："101 Formulaic Alphas" 中广泛使用 SignedPower 算子
- **Tulchinsky (2019)**："Finding Alphas" (2nd Ed.) 第4章详细论述了幂变换在因子精炼中的作用
- **实证发现**：在本项目的25种优化方法中，有符号幂变换的G类方法综合得分最高（平均0.0646），远超其他方法类别

### 5. 两轮优化策略

- **第一轮**：从现有311维因子库中选取IC排名靠前且分布适合幂变换的因子，按因子类型匹配SP15/SP20/SP05/SP03
- **第二轮**：基于第一轮发现（有符号振荡型响应最好），拓展因子库覆盖多周期EMA乖离、MOM/ROC、趋势偏离、振荡器、量价交互、跨周期价差、布林带位置等新类别，全部应用SP15和SP20

## 第一轮 Signed Power 优化因子 — 公式与介绍

### SP15 温和放大组 ($p=1.5$) — 14个因子

适用于中心在0附近的有符号振荡型因子，通过 $\text{sign}(x) \cdot |x|^{1.5}$ 温和放大极端信号。

---

**1. SP15_Intrabar_VWAP_Dev — VWAP偏离^1.5**

- **原始因子**：$\text{VWAP\_Dev} = \dfrac{C - \text{OHLC4}}{\text{OHLC4}}$，其中 $\text{OHLC4} = \frac{O+H+L+C}{4}$
- **物理含义**：收盘价相对于K线内均价的偏离程度，正值表示收盘价偏高于均价
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{VWAP\_Dev}) \cdot |\text{VWAP\_Dev}|^{1.5}$
- **预期效果**：放大极端VWAP偏离日的信号权重，温和抑制微弱偏离噪音

---

**2. SP15_Order_Imbalance_Px — 订单失衡^1.5**

- **原始因子**：$\text{OI} = \dfrac{C_t - C_{t-1}}{\ln(\max(V_t, 1))}$
- **物理含义**：单位对数成交量的价格变化，衡量买卖压力失衡
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{OI}) \cdot |\text{OI}|^{1.5}$

---

**3. SP15_ema_ret_60M — EMA60乖离^1.5**

- **原始因子**：$\text{ema\_ret\_60} = \dfrac{C - \text{EMA}_{60}(C)}{\text{EMA}_{60}(C)}$
- **物理含义**：价格偏离60周期指数均线的程度，正值表示价格高于均线（多头趋势）
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{ema\_ret\_60}) \cdot |\text{ema\_ret\_60}|^{1.5}$
- **预期效果**：放大趋势极端乖离日的信号，提升趋势跟踪类因子的IC/IR

---

**4. SP15_roc_5M — 5期ROC^1.5**

- **原始因子**：$\text{roc\_5} = \dfrac{C_t - C_{t-5}}{C_{t-5}}$
- **物理含义**：5周期价格变化率，衡量短期动量
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{roc\_5}) \cdot |\text{roc\_5}|^{1.5}$

---

**5. SP15_roc_60M — 60期ROC^1.5**

- **原始因子**：$\text{roc\_60} = \dfrac{C_t - C_{t-60}}{C_{t-60}}$
- **物理含义**：60周期价格变化率，衡量中长期动量
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{roc\_60}) \cdot |\text{roc\_60}|^{1.5}$

---

**6. SP15_macd_60M — MACD^1.5**

- **原始因子**：$\text{MACD} = \text{EMA}_{12}(C) - \text{EMA}_{26}(C)$
- **物理含义**：短期与长期指数均线的差值，经典趋势跟踪指标
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{MACD}) \cdot |\text{MACD}|^{1.5}$

---

**7. SP15_ppo_60M — PPO^1.5**

- **原始因子**：$\text{PPO} = \dfrac{\text{EMA}_{12}(C) - \text{EMA}_{26}(C)}{\text{EMA}_{26}(C)} \times 100$
- **物理含义**：MACD的百分比版本，消除价格量纲影响
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{PPO}) \cdot |\text{PPO}|^{1.5}$

---

**8. SP15_trix_60M — TRIX^1.5**

- **原始因子**：$\text{TRIX} = \text{ROC}_1(\text{EMA}_3(\text{EMA}_3(\text{EMA}_3(C))))$，即三重指数平滑的变化率
- **物理含义**：过滤高频噪声后的趋势加速度
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{TRIX}) \cdot |\text{TRIX}|^{1.5}$

---

**9. SP15_vpats_60M — VPATS^1.5**

- **原始因子**：$\text{VPATS} = \dfrac{\text{HMA} - \text{KAMA}}{\text{KAMA}} \cdot e^{-10 \cdot \text{NATR}}$
- **物理含义**：HMA与KAMA的偏离度，经归一化ATR衰减（高波动时信号被抑制）
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{VPATS}) \cdot |\text{VPATS}|^{1.5}$

---

**10. SP15_cci_60M — CCI^1.5**

- **原始因子**：$\text{CCI} = \dfrac{\text{TP} - \text{SMA}_{60}(\text{TP})}{0.015 \cdot \text{MeanDeviation}}$，其中 $\text{TP} = \frac{H+L+C}{3}$
- **物理含义**：典型价格偏离均值的统计标准化度量
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{CCI}) \cdot |\text{CCI}|^{1.5}$

---

**11. SP15_rsi_centered_60M — RSI中心化^1.5**

- **原始因子**：$\text{RSI\_centered} = \dfrac{\text{RSI}_{60} - 50}{50}$
- **物理含义**：RSI减去50后缩放到[-1,1]，正值表示超买，负值表示超卖
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{RSI\_centered}) \cdot |\text{RSI\_centered}|^{1.5}$

---

**12. SP15_cmf_60M — CMF^1.5**

- **原始因子**：$\text{MFM} = \dfrac{(C-L)-(H-C)}{H-L}$，$\text{CMF}_{60} = \dfrac{\sum_{i=t-59}^{t} \text{MFM}_i \cdot V_i}{\sum_{i=t-59}^{t} V_i}$
- **物理含义**：60周期成交量加权的Chaikin资金流向指标
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{CMF}) \cdot |\text{CMF}|^{1.5}$

---

**13. SP15_Close_Position — 收盘位置中心化^1.5**

- **原始因子**：$\text{ClosePos} = \dfrac{C-L}{H-L} - 0.5$（中心化至[-0.5, 0.5]）
- **物理含义**：收盘价在K线内的相对位置，正值表示收盘偏高位
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{ClosePos}) \cdot |\text{ClosePos}|^{1.5}$

---

**14. SP15_Upper_Shadow_Frac — 上影线占比^1.5**

- **原始因子**：$\text{UpperShadow} = \dfrac{H - \max(O, C)}{H - L}$
- **物理含义**：上影线占整根K线的比例，衡量上方阻力
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{UpperShadow}) \cdot |\text{UpperShadow}|^{1.5}$

---

### SP20 强力放大组 ($p=2.0$) — 8个有符号因子

通过 $\text{sign}(x) \cdot x^2$（等价于 $x \cdot |x|$）实现平方级放大。

---

**15. SP20_Intrabar_VWAP_Dev — VWAP偏离^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{VWAP\_Dev}) \cdot |\text{VWAP\_Dev}|^{2.0}$

---

**16. SP20_LogRet_1 — 对数收益率^2.0**

- **原始因子**：$\text{LogRet} = \ln\left(\dfrac{C_t}{C_{t-1}}\right)$
- **物理含义**：单期对数收益率
- **SignedPower变换**：$\text{SP20} = \text{sign}(\text{LogRet}) \cdot |\text{LogRet}|^{2.0}$
- **预期效果**：$p=2.0$ 强力放大极端涨跌日的信号

---

**17. SP20_Order_Imbalance_Px — 订单失衡^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{OI}) \cdot |\text{OI}|^{2.0}$

---

**18. SP20_ema_ret_60M — EMA60乖离^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{ema\_ret\_60}) \cdot |\text{ema\_ret\_60}|^{2.0}$

---

**19. SP20_roc_5M — 5期ROC^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{roc\_5}) \cdot |\text{roc\_5}|^{2.0}$

---

**20. SP20_roc_60M — 60期ROC^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{roc\_60}) \cdot |\text{roc\_60}|^{2.0}$

---

**21. SP20_macd_60M — MACD^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{MACD}) \cdot |\text{MACD}|^{2.0}$

---

**22. SP20_trix_60M — TRIX^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{TRIX}) \cdot |\text{TRIX}|^{2.0}$

---

### SP05 平方根压缩组 ($p=0.5$) — 6个正值因子

通过 $\max(x, 0)^{0.5}$（即 $\sqrt{x}$）压缩重尾正值因子的极端值。

---

**23. SP05_GK_Volatility_1M — GK波动率^0.5**

- **原始因子**：$\sigma_{GK} = \max\left(0.5(\ln\frac{H}{L})^2 - (2\ln 2 - 1)(\ln\frac{O}{C})^2, \; 0\right)$
- **物理含义**：Garman-Klass波动率估计量，利用OHLC信息的高效波动率估计
- **幂压缩变换**：$\text{SP05} = (\sigma_{GK})^{0.5}$
- **预期效果**：压缩波动率极端飙升日的影响，使波动率信号更稳定

---

**24. SP05_Log_HL_Range — 对数极差^0.5**

- **原始因子**：$\text{LogHL} = \ln\left(\dfrac{\max(H, L)}{\min(H, L)}\right) \approx \ln\left(\dfrac{H}{L}\right)$
- **物理含义**：对数高低价极差，衡量日内波动幅度
- **幂压缩变换**：$\text{SP05} = (\text{LogHL})^{0.5}$

---

**25. SP05_natr_60M — 归一化ATR^0.5**

- **原始因子**：$\text{NATR}_{60} = \dfrac{\text{ATR}_{60}}{C}$
- **物理含义**：60周期平均真实波幅除以收盘价，消除价格量纲
- **幂压缩变换**：$\text{SP05} = (\text{NATR}_{60})^{0.5}$

---

**26. SP05_adx_60M — ADX^0.5**

- **原始因子**：$\text{ADX}_{60}$（Wilder平均趋向指数）
- **物理含义**：衡量趋势强度的非方向性指标
- **幂压缩变换**：$\text{SP05} = (\text{ADX}_{60})^{0.5}$

---

**27. SP05_Realized_Vol_5M — 5期实现波动率^0.5**

- **原始因子**：$\text{RV}_5 = \sqrt{\sum_{i=0}^{4} (\ln\frac{C_{t-i}}{C_{t-i-1}})^2}$
- **物理含义**：近5期的实现波动率
- **幂压缩变换**：$\text{SP05} = (\text{RV}_5)^{0.5}$

---

**28. SP05_Log_HL_Range_Roll60 — 60期滚动LogHL均值^0.5**

- **原始因子**：$\text{LogHL\_60} = \dfrac{1}{60}\sum_{i=0}^{59} \text{LogHL}_{t-i}$
- **物理含义**：对数极差的60周期滚动均值，衡量长期波动水平
- **幂压缩变换**：$\text{SP05} = (\text{LogHL\_60})^{0.5}$

---

### SP03 强力压缩组 ($p=0.3$) — 5个正值因子

通过 $\max(x, 0)^{0.3}$ 对极度重尾的正值因子施加更强的压缩。

---

**29. SP03_GK_Volatility_1M — GK波动率^0.3**

- **公式**：$\text{SP03} = (\sigma_{GK})^{0.3}$

---

**30. SP03_Log_HL_Range — 对数极差^0.3**

- **公式**：$\text{SP03} = (\text{LogHL})^{0.3}$

---

**31. SP03_natr_60M — 归一化ATR^0.3**

- **公式**：$\text{SP03} = (\text{NATR}_{60})^{0.3}$

---

**32. SP03_Jump_Ratio_60M — 跳跃比率^0.3**

- **原始因子**：$\text{JR} = \max\left(\dfrac{\text{RV}_{60} - \text{BPV}_{60}}{\text{RV}_{60}}, \; 0\right)$
  - $\text{RV}_{60} = \sum_{i=0}^{59} r_{t-i}^2$（实现方差）
  - $\text{BPV}_{60} = \frac{\pi}{2} \sum_{i=1}^{59} |r_{t-i}| \cdot |r_{t-i-1}|$（双幂变差）
- **物理含义**：衡量跳跃成分在总波动中的占比，非负值
- **幂压缩变换**：$\text{SP03} = (\text{JR})^{0.3}$

---

**33. SP03_Amihud_60M — Amihud非流动性^0.3**

- **原始因子**：$\text{Amihud}_{60} = \dfrac{1}{60}\sum_{i=0}^{59} \dfrac{|r_{t-i}|}{V_{t-i} \cdot C_{t-i}}$
- **物理含义**：单位成交金额引起的价格变化，衡量非流动性，具有极度重尾特征
- **幂压缩变换**：$\text{SP03} = (\text{Amihud}_{60})^{0.3}$

---

### SP20 正值有界型 ($[0,1]$ 区间，$p=2.0$) — 2个因子

---

**34. SP20_Close_Position — 收盘位置^2**

- **原始因子**：$\text{ClosePos} = \dfrac{C-L}{H-L}$（值域 $[0,1]$）
- **物理含义**：收盘价在K线内的相对位置
- **幂变换**：$\text{SP20} = (\text{ClosePos})^{2.0}$
- **预期效果**：放大高位收盘信号（接近1时），抑制低位收盘信号（接近0时）

---

**35. SP20_Body_Fraction — 实体占比^2**

- **原始因子**：$\text{BodyFrac} = \dfrac{|C-O|}{H-L}$（值域 $[0,1]$）
- **物理含义**：K线实体占整根K线的比例
- **幂变换**：$\text{SP20} = (\text{BodyFrac})^{2.0}$

In [ ]:
# ==========================================
# Signed Power 对现有因子库优化 — 35个优化因子
# ==========================================
import os, time, gc
import numpy as np
import pandas as pd
import talib
import bottleneck as bn
from numpy.lib.stride_tricks import sliding_window_view
from pathlib import Path

BASE_DIR = Path(r"C:\Users\wangx\Desktop\华东")
RAW_DATA_DIR = BASE_DIR / "train_dataset" / "train_dataset"
OUTPUT_CSV2 = BASE_DIR / "SignedPower_Optimized_IC_Evaluation.csv"
DATASET_COUNT = 30

# 35个Signed Power优化因子
SP_FACTOR_NAMES = [
    # === 有符号振荡型 — p=1.5 (温和放大) ===
    "SP15_Intrabar_VWAP_Dev",      # VWAP偏离^1.5
    "SP15_Order_Imbalance_Px",     # 订单失衡^1.5
    "SP15_ema_ret_60M",            # EMA乖离^1.5
    "SP15_roc_5M",                 # 5期ROC^1.5
    "SP15_roc_60M",                # 60期ROC^1.5
    "SP15_macd_60M",               # MACD^1.5
    "SP15_ppo_60M",                # PPO^1.5
    "SP15_trix_60M",               # TRIX^1.5
    "SP15_vpats_60M",              # VPATS^1.5
    "SP15_cci_60M",                # CCI^1.5
    "SP15_rsi_centered_60M",       # RSI(中心化)^1.5
    "SP15_cmf_60M",                # CMF^1.5
    "SP15_Close_Position",         # 收盘位置(中心化)^1.5
    "SP15_Upper_Shadow_Frac",      # 上影线^1.5
    # === 有符号振荡型 — p=2.0 (强力放大) ===
    "SP20_Intrabar_VWAP_Dev",      # VWAP偏离^2.0
    "SP20_LogRet_1",               # 对数收益率^2.0
    "SP20_Order_Imbalance_Px",     # 订单失衡^2.0
    "SP20_ema_ret_60M",            # EMA乖离^2.0
    "SP20_roc_5M",                 # 5期ROC^2.0
    "SP20_roc_60M",                # 60期ROC^2.0
    "SP20_macd_60M",               # MACD^2.0
    "SP20_trix_60M",               # TRIX^2.0
    # === 正值重尾型 — p=0.5 (平方根压缩) ===
    "SP05_GK_Volatility_1M",       # GK波动率^0.5
    "SP05_Log_HL_Range",           # 对数极差^0.5
    "SP05_natr_60M",               # 归一化ATR^0.5
    "SP05_adx_60M",                # ADX^0.5
    "SP05_Realized_Vol_5M",        # 5期实现波动率^0.5
    "SP05_Log_HL_Range_Roll60",    # 60期滚动LogHL均值^0.5
    # === 正值重尾型 — p=0.3 (强力压缩) ===
    "SP03_GK_Volatility_1M",       # GK波动率^0.3
    "SP03_Log_HL_Range",           # 对数极差^0.3
    "SP03_natr_60M",               # 归一化ATR^0.3
    "SP03_Jump_Ratio_60M",         # 跳跃比例^0.3
    "SP03_Amihud_60M",             # Amihud非流动性^0.3
    # === 特殊: 正值[0,1]区间因子 — p=2.0 ===
    "SP20_Close_Position",         # 收盘位置^2
    "SP20_Body_Fraction",          # 实体占比^2
]
FACTOR_DIM = len(SP_FACTOR_NAMES)
print(f"共 {FACTOR_DIM} 个Signed Power优化因子")
print(f"  SP15 (温和放大): 14个 | SP20 (强力放大): 10个")
print(f"  SP05 (平方根压缩): 6个 | SP03 (强力压缩): 5个")

# ==========================================
# 基础因子计算 (复现因子库中的Top IC因子)
# ==========================================
def compute_base_factors_extended(O, H, L, C, V):
    eps = 1e-8
    C_64 = C.astype(np.float64); H_64 = H.astype(np.float64)
    L_64 = L.astype(np.float64); V_64 = V.astype(np.float64)
    O_64 = O.astype(np.float64)

    # Level 1: 单K线微观形态
    log_hl = np.log(np.maximum(H_64, L_64+eps) / np.maximum(L_64, eps))
    log_oc = np.log(np.maximum(O_64, C_64+eps) / np.maximum(C_64, eps))
    gk_vol = np.maximum(0.5*log_hl**2 - (2*np.log(2)-1)*log_oc**2, 0)
    log_hl_range = log_hl

    ohlc4 = (O_64 + H_64 + L_64 + C_64) / 4.0
    vwap_dev = (C_64 - ohlc4) / (ohlc4 + eps)

    delta_c = C_64 - np.roll(C_64, 1); delta_c[0] = 0.0
    order_imb = delta_c / (np.log(np.maximum(V_64, 1)) + eps)

    hl_r = H_64 - L_64 + eps
    close_pos = (C_64 - L_64) / hl_r
    upper_shadow = (H_64 - np.maximum(O_64, C_64)) / hl_r
    body_frac = np.abs(C_64 - O_64) / hl_r

    log_ret = np.log(np.maximum(C_64, eps)) - np.log(np.maximum(np.roll(C_64, 1), eps))
    log_ret[0] = 0.0

    # Level 2: 滚动窗口技术指标
    p60 = 60
    ema60 = talib.EMA(C_64, timeperiod=p60)
    ema60 = np.nan_to_num(ema60, nan=C_64[0])
    ema_ret_60 = (C_64 - ema60) / (ema60 + eps)

    roc_5 = (C_64 - np.roll(C_64, 5)) / (np.roll(C_64, 5) + eps); roc_5[:5] = 0.0
    roc_60 = (C_64 - np.roll(C_64, 60)) / (np.roll(C_64, 60) + eps); roc_60[:60] = 0.0

    trix_60 = talib.TRIX(C_64, timeperiod=p60)
    trix_60 = np.nan_to_num(trix_60, nan=0.0)

    ema12 = talib.EMA(C_64, timeperiod=12); ema26 = talib.EMA(C_64, timeperiod=26)
    ema12 = np.nan_to_num(ema12, nan=C_64[0]); ema26 = np.nan_to_num(ema26, nan=C_64[0])
    macd_60 = ema12 - ema26

    ppo_60 = talib.PPO(C_64, fastperiod=12, slowperiod=26)
    ppo_60 = np.nan_to_num(ppo_60, nan=0.0)

    rsi_60 = talib.RSI(C_64, timeperiod=p60)
    rsi_centered = np.nan_to_num((rsi_60 - 50.0) / 50.0, nan=0.0)

    cci_60 = talib.CCI(H_64, L_64, C_64, timeperiod=p60)
    cci_60 = np.nan_to_num(cci_60, nan=0.0)

    # VPATS
    atr60 = talib.ATR(H_64, L_64, C_64, timeperiod=p60)
    natr_60 = atr60 / (C_64 + eps)
    kama60 = talib.KAMA(C_64, timeperiod=p60)
    half_p = max(2, p60//2); sqrt_p = max(2, int(np.sqrt(p60)))
    wma_h = talib.WMA(C_64, timeperiod=half_p); wma_f = talib.WMA(C_64, timeperiod=p60)
    hma60 = talib.WMA(2*wma_h - wma_f, timeperiod=sqrt_p)
    vpats_60 = np.nan_to_num(((hma60 - kama60)/(kama60+eps)) * np.exp(-10.0*natr_60), nan=0.0)

    adx_60 = talib.ADX(H_64, L_64, C_64, timeperiod=p60)
    adx_60 = np.nan_to_num(adx_60, nan=0.0)

    # CMF
    mf_mult = ((C_64 - L_64) - (H_64 - C_64)) / (hl_r + eps)
    mf_vol = mf_mult * V_64
    cmf_60 = bn.move_sum(mf_vol, window=p60, min_count=1) / (bn.move_sum(V_64, window=p60, min_count=1) + eps)

    # Amihud
    amihud_60 = bn.move_mean(np.abs(log_ret) / (V_64*C_64 + eps), window=p60, min_count=1)

    # RV
    rv_5 = np.sqrt(bn.move_sum(log_ret**2, window=5, min_count=1))

    # Jump Ratio
    abs_ret = np.abs(log_ret.astype(np.float64))
    bpv60 = (np.pi/2) * bn.move_sum(np.roll(abs_ret,1)*abs_ret, window=p60, min_count=1)
    rv60 = bn.move_sum(log_ret.astype(np.float64)**2, window=p60, min_count=1)

    # Rolling Log_HL
    log_hl_60 = bn.move_mean(log_hl_range, window=p60, min_count=1)

    return {
        'gk_vol': gk_vol.astype(np.float32), 'log_hl_range': log_hl_range.astype(np.float32),
        'vwap_dev': vwap_dev.astype(np.float32), 'order_imb': order_imb.astype(np.float32),
        'close_pos': close_pos.astype(np.float32), 'upper_shadow': upper_shadow.astype(np.float32),
        'body_frac': body_frac.astype(np.float32), 'log_ret': log_ret.astype(np.float32),
        'ema_ret_60': ema_ret_60.astype(np.float32), 'roc_5': roc_5.astype(np.float32),
        'roc_60': roc_60.astype(np.float32), 'trix_60': trix_60.astype(np.float32),
        'macd_60': macd_60.astype(np.float32), 'ppo_60': ppo_60.astype(np.float32),
        'rsi_centered': rsi_centered.astype(np.float32), 'cci_60': cci_60.astype(np.float32),
        'vpats_60': vpats_60.astype(np.float32), 'natr_60': natr_60.astype(np.float32),
        'adx_60': adx_60.astype(np.float32), 'cmf_60': cmf_60.astype(np.float32),
        'amihud_60': amihud_60.astype(np.float32), 'rv_5': rv_5.astype(np.float32),
        'log_hl_60': log_hl_60.astype(np.float32), 'rv60': rv60.astype(np.float32),
        'bpv60': bpv60.astype(np.float32),
    }


def signed_power(x, p):
    return np.sign(x) * (np.abs(x) ** p)

def power_only(x, p):
    return np.maximum(x, 0) ** p


def generate_sp_optimized_factors(O, H, L, C, V):
    bf = compute_base_factors_extended(O, H, L, C, V)
    eps = 1e-8
    f = []

    # --- SP15: signed, p=1.5 (14) ---
    f.append(signed_power(bf['vwap_dev'], 1.5))
    f.append(signed_power(bf['order_imb'], 1.5))
    f.append(signed_power(bf['ema_ret_60'], 1.5))
    f.append(signed_power(bf['roc_5'], 1.5))
    f.append(signed_power(bf['roc_60'], 1.5))
    f.append(signed_power(bf['macd_60'], 1.5))
    f.append(signed_power(bf['ppo_60'], 1.5))
    f.append(signed_power(bf['trix_60'], 1.5))
    f.append(signed_power(bf['vpats_60'], 1.5))
    f.append(signed_power(bf['cci_60'], 1.5))
    f.append(signed_power(bf['rsi_centered'], 1.5))
    f.append(signed_power(bf['cmf_60'], 1.5))
    f.append(signed_power(bf['close_pos'] - 0.5, 1.5))
    f.append(signed_power(bf['upper_shadow'], 1.5))

    # --- SP20: signed, p=2.0 (8) ---
    f.append(signed_power(bf['vwap_dev'], 2.0))
    f.append(signed_power(bf['log_ret'], 2.0))
    f.append(signed_power(bf['order_imb'], 2.0))
    f.append(signed_power(bf['ema_ret_60'], 2.0))
    f.append(signed_power(bf['roc_5'], 2.0))
    f.append(signed_power(bf['roc_60'], 2.0))
    f.append(signed_power(bf['macd_60'], 2.0))
    f.append(signed_power(bf['trix_60'], 2.0))

    # --- SP05: positive, p=0.5 (6) ---
    f.append(power_only(bf['gk_vol'], 0.5))
    f.append(power_only(bf['log_hl_range'], 0.5))
    f.append(power_only(bf['natr_60'], 0.5))
    f.append(power_only(bf['adx_60'], 0.5))
    f.append(power_only(bf['rv_5'], 0.5))
    f.append(power_only(bf['log_hl_60'], 0.5))

    # --- SP03: positive, p=0.3 (5) ---
    f.append(power_only(bf['gk_vol'], 0.3))
    f.append(power_only(bf['log_hl_range'], 0.3))
    f.append(power_only(bf['natr_60'], 0.3))
    jump_ratio = np.maximum((bf['rv60'] - bf['bpv60']) / (bf['rv60'] + eps), 0)
    f.append(power_only(jump_ratio, 0.3))
    f.append(power_only(bf['amihud_60'], 0.3))

    # --- SP20: [0,1] bounded, p=2.0 (2) ---
    f.append(power_only(bf['close_pos'], 2.0))
    f.append(power_only(bf['body_frac'], 2.0))

    return np.column_stack(f).astype(np.float32)


def calculate_pearson_ic(factors, label):
    vm = ~np.isnan(label); y = label[vm]; X = factors[vm, :]
    nm = np.isnan(X); X = np.where(nm,0,X)
    vc = (~nm).sum(axis=0); vc = np.where(vc==0,1e-8,vc)
    ym=np.mean(y); xm=X.sum(axis=0)/vc
    yc=y-ym; xc=np.where(nm,0,X-xm)
    cov=np.dot(yc,xc)/vc
    ys=np.sqrt(np.mean(yc**2)); xs=np.sqrt(np.sum(xc**2,axis=0)/vc)
    ic=cov/(ys*xs+1e-8); ic=np.where(vc<len(y)*0.1,0,ic)
    return ic


def evaluate_sp_optimized_factors():
    print("="*70)
    print("[Signed Power] 对现有因子库Top IC因子的有符号幂变换优化")
    print("="*70)

    ic5 = np.zeros((DATASET_COUNT, FACTOR_DIM), dtype=np.float32)
    ic60 = np.zeros((DATASET_COUNT, FACTOR_DIM), dtype=np.float32)
    success = 0; t0 = time.time()

    for i in range(DATASET_COUNT):
        rp = RAW_DATA_DIR / f"dataset{i}_train_ohlcv.npy"
        if not rp.exists(): continue
        try:
            raw = np.load(rp).astype(np.float32)
            if raw.shape[1] != 8: continue
            O=raw[:,1]; H=raw[:,2]; L=raw[:,3]; C=raw[:,4]; V=raw[:,5]
            spf = generate_sp_optimized_factors(O, H, L, C, V)
            ic5[i,:] = calculate_pearson_ic(spf, raw[:,6])
            ic60[i,:] = calculate_pearson_ic(spf, raw[:,7])
            success += 1
            if success % 5 == 0:
                print(f"  已完成 {success}/{DATASET_COUNT} (耗时{(time.time()-t0)/60:.1f}分)...")
            del raw, spf; gc.collect()
        except Exception as e:
            print(f"  dataset{i} 失败: {e}")

    elapsed = (time.time()-t0)/60
    print(f"\n[完成] {success}/{DATASET_COUNT}, 耗时 {elapsed:.2f} 分钟")

    vic5=ic5[:success]; vic60=ic60[:success]
    m5=np.nanmean(vic5,0); m60=np.nanmean(vic60,0)
    s5=np.nanstd(vic5,0); s60=np.nanstd(vic60,0)
    ir5=m5/(s5+1e-8); ir60=m60/(s60+1e-8)
    sc5=np.abs(m5)*np.abs(ir5); sc60=np.abs(m60)*np.abs(ir60)

    def get_pt(name):
        if 'SP15' in name: return 'SP15_温和放大_p1.5'
        if 'SP20' in name: return 'SP20_强力放大_p2.0'
        if 'SP05' in name: return 'SP05_平方根压缩_p0.5'
        if 'SP03' in name: return 'SP03_强力压缩_p0.3'
        return 'Unknown'

    df = pd.DataFrame({
        'Factor_Name': SP_FACTOR_NAMES,
        'Power_Type': [get_pt(n) for n in SP_FACTOR_NAMES],
        'Ret5_Mean_IC': m5, 'Ret5_Std_IC': s5, 'Ret5_IC_IR': ir5,
        'Ret60_Mean_IC': m60, 'Ret60_Std_IC': s60, 'Ret60_IC_IR': ir60,
        'Ret5_Score': sc5, 'Ret60_Score': sc60,
        'Comprehensive_Score': sc5 + sc60
    })

    print("\n" + "="*70)
    print("[Ret5 短线] Top 10 Signed Power优化因子:")
    print("="*70)
    t5 = df.sort_values('Ret5_Score', ascending=False).head(10)
    print(t5[['Factor_Name', 'Power_Type', 'Ret5_Mean_IC', 'Ret5_IC_IR', 'Ret5_Score']].to_string(index=False))

    print("\n" + "="*70)
    print("[Ret60 长线] Top 10 Signed Power优化因子:")
    print("="*70)
    t60 = df.sort_values('Ret60_Score', ascending=False).head(10)
    print(t60[['Factor_Name', 'Power_Type', 'Ret60_Mean_IC', 'Ret60_IC_IR', 'Ret60_Score']].to_string(index=False))

    print("\n" + "="*70)
    print("[综合排名] 全部Signed Power优化因子:")
    print("="*70)
    dfs = df.sort_values('Comprehensive_Score', ascending=False)
    print(dfs[['Factor_Name', 'Power_Type', 'Ret5_Mean_IC', 'Ret60_Mean_IC', 'Comprehensive_Score']].to_string(index=False))

    print("\n" + "="*70)
    print("[幂次类型汇总]")
    print("="*70)
    for pt in sorted(df['Power_Type'].unique()):
        sub = df[df['Power_Type'] == pt].dropna(subset=['Comprehensive_Score'])
        if len(sub) == 0: continue
        print(f"  {pt:35s}  平均={sub['Comprehensive_Score'].mean():.6f}  最高={sub['Comprehensive_Score'].max():.6f}  ({sub.loc[sub['Comprehensive_Score'].idxmax(),'Factor_Name']})")

    # 与原始因子对比
    print("\n" + "="*70)
    print("[优化前后对比] 原始IC vs SignedPower优化后IC (Ret5 |IC|)")
    print("="*70)
    orig_ref = {
        'Intrabar_VWAP_Dev': (0.0352, 'SP15_Intrabar_VWAP_Dev'),
        'Order_Imbalance_Px': (0.0232, 'SP15_Order_Imbalance_Px'),
        'GK_Volatility_1M': (0.0314, 'SP05_GK_Volatility_1M'),
        'Log_HL_Range': (0.0140, 'SP05_Log_HL_Range'),
        'LogRet_1': (0.0319, 'SP20_LogRet_1'),
        'ema_ret_60M': (0.0319, 'SP15_ema_ret_60M'),
    }
    for base, (orig_ic, sp_name) in orig_ref.items():
        row = df[df['Factor_Name'] == sp_name]
        if len(row) > 0:
            new_ic = abs(row.iloc[0]['Ret5_Mean_IC'])
            change = (new_ic - orig_ic) / orig_ic * 100
            print(f"  {base:25s} -> {sp_name:30s}  |IC|: {orig_ic:.4f} -> {new_ic:.4f} ({change:+.1f}%)")

    df.to_csv(OUTPUT_CSV2, index=False, encoding='utf-8-sig')
    print(f"\n[保存] {OUTPUT_CSV2}")
    print("="*70)
    return df

if __name__ == "__main__":
    df_sp_result = evaluate_sp_optimized_factors()


## 第二轮 Signed Power 优化因子 — 公式与介绍

基于第一轮实证发现（有符号振荡型因子对 SP15/SP20 响应最好），本轮拓展因子库覆盖更多类别，全部应用 SP15 ($p=1.5$) 和 SP20 ($p=2.0$) 两种有符号幂变换。

---

### 一、多周期EMA乖离 (4个)

通过不同周期的EMA乖离对价格趋势偏离程度进行多尺度刻画。

---

**1. SP15_ema_ret_5M — EMA5乖离^1.5**

- **原始因子**：$\text{ema\_ret\_5} = \dfrac{C - \text{EMA}_{5}(C)}{\text{EMA}_{5}(C)}$
- **物理含义**：价格偏离5周期指数均线的程度，捕捉超短期趋势偏离
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{ema\_ret\_5}) \cdot |\text{ema\_ret\_5}|^{1.5}$

**2. SP20_ema_ret_5M — EMA5乖离^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{ema\_ret\_5}) \cdot |\text{ema\_ret\_5}|^{2.0}$

---

**3. SP15_ema_ret_15M — EMA15乖离^1.5**

- **原始因子**：$\text{ema\_ret\_15} = \dfrac{C - \text{EMA}_{15}(C)}{\text{EMA}_{15}(C)}$
- **物理含义**：价格偏离15周期指数均线的程度，捕捉中短期趋势偏离
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{ema\_ret\_15}) \cdot |\text{ema\_ret\_15}|^{1.5}$

**4. SP20_ema_ret_15M — EMA15乖离^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{ema\_ret\_15}) \cdot |\text{ema\_ret\_15}|^{2.0}$

---

### 二、多周期MOM/ROC (6个)

填补第一轮中ROC_5M和ROC_60M之间的周期断层，构建完整的动量周期谱。

---

**5. SP15_mom_5M — 5期价格动量^1.5**

- **原始因子**：$\text{mom\_5} = C_t - C_{t-5}$（绝对价格变化）
- **物理含义**：5周期绝对价格变化，与ROC不同（保留价格量纲）
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{mom\_5}) \cdot |\text{mom\_5}|^{1.5}$

**6. SP20_mom_5M — 5期价格动量^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{mom\_5}) \cdot |\text{mom\_5}|^{2.0}$

---

**7. SP15_mom_60M — 60期价格动量^1.5**

- **原始因子**：$\text{mom\_60} = C_t - C_{t-60}$
- **物理含义**：60周期绝对价格变化，中长期动量
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{mom\_60}) \cdot |\text{mom\_60}|^{1.5}$

**8. SP20_mom_60M — 60期价格动量^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{mom\_60}) \cdot |\text{mom\_60}|^{2.0}$

---

**9. SP15_roc_15M — 15期ROC^1.5**

- **原始因子**：$\text{roc\_15} = \dfrac{C_t - C_{t-15}}{C_{t-15}}$
- **物理含义**：15周期价格变化率，中短期动量（介于5M和60M之间）
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{roc\_15}) \cdot |\text{roc\_15}|^{1.5}$

**10. SP20_roc_15M — 15期ROC^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{roc\_15}) \cdot |\text{roc\_15}|^{2.0}$

---

### 三、趋势偏离类 (6个)

计算价格与各类自适应均线（HMA/KAMA/Ichimoku）的偏离度，捕捉均值回归机会。

---

**11. SP15_HMA_Deviation — HMA偏离^1.5**

- **原始因子**：$\text{HMA\_Dev} = \dfrac{\text{HMA}_{60} - C}{C}$
  - $\text{HMA} = \text{WMA}_{\sqrt{N}}(2 \cdot \text{WMA}_{N/2}(C) - \text{WMA}_{N}(C))$（Hull移动平均）
- **物理含义**：Hull移动平均与当前价格的相对偏离，正值表示价格低于HMA（潜在反弹信号）
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{HMA\_Dev}) \cdot |\text{HMA\_Dev}|^{1.5}$

**12. SP20_HMA_Deviation — HMA偏离^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{HMA\_Dev}) \cdot |\text{HMA\_Dev}|^{2.0}$

---

**13. SP15_KAMA_Deviation — KAMA偏离^1.5**

- **原始因子**：$\text{KAMA\_Dev} = \dfrac{\text{KAMA}_{60} - C}{C}$
  - KAMA（Kaufman自适应移动平均）根据市场效率比率动态调整平滑系数
- **物理含义**：自适应均线与价格的偏离，趋势市时KAMA紧跟价格，震荡市时KAMA走平
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{KAMA\_Dev}) \cdot |\text{KAMA\_Dev}|^{1.5}$

**14. SP20_KAMA_Deviation — KAMA偏离^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{KAMA\_Dev}) \cdot |\text{KAMA\_Dev}|^{2.0}$

---

**15. SP15_Ichimoku_Premium — Ichimoku溢价^1.5**

- **原始因子**：$\text{Ichimoku\_Prem} = \dfrac{C - \frac{\text{HH}_{30} + \text{LL}_{30}}{2}}{\frac{\text{HH}_{30} + \text{LL}_{30}}{2}}$
  - $\text{HH}_{30} = \max_{i=0}^{29} H_{t-i}$，$\text{LL}_{30} = \min_{i=0}^{29} L_{t-i}$
- **物理含义**：价格相对于一目均衡表云带中轨的偏离
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{Ichimoku\_Prem}) \cdot |\text{Ichimoku\_Prem}|^{1.5}$

**16. SP20_Ichimoku_Premium — Ichimoku溢价^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{Ichimoku\_Prem}) \cdot |\text{Ichimoku\_Prem}|^{2.0}$

---

### 四、振荡器类 (8个)

将TA-Lib经典振荡器中心化/归一化后应用SignedPower变换，增强极端读数的信号强度。

---

**17. SP15_WILLR_60M — Williams %R^1.5**

- **原始因子**：$\text{WILLR}_{60} = \dfrac{\text{HH}_{60} - C}{\text{HH}_{60} - \text{LL}_{60}} \times (-100)$，缩放到 $[-1, 0]$
- **物理含义**：收盘价在60周期最高-最低范围内的相对位置（反转版），接近0=超买，接近-1=超卖
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{WILLR}) \cdot |\text{WILLR}|^{1.5}$

**18. SP20_WILLR_60M — Williams %R^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{WILLR}) \cdot |\text{WILLR}|^{2.0}$

---

**19. SP15_CMO_60M — Chande动量振荡器^1.5**

- **原始因子**：$\text{CMO}_{60} = \dfrac{\sum \text{Up} - \sum \text{Down}}{\sum \text{Up} + \sum \text{Down}}$，缩放到 $[-1, 1]$
- **物理含义**：类似RSI但使用原始涨跌和而非平均涨跌，对极端动量更敏感
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{CMO}) \cdot |\text{CMO}|^{1.5}$

**20. SP20_CMO_60M — Chande动量振荡器^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{CMO}) \cdot |\text{CMO}|^{2.0}$

---

**21. SP15_BOP_EMA_60M — BOP平滑^1.5**

- **原始因子**：$\text{BOP} = \dfrac{C - O}{H - L}$，$\text{BOP\_EMA}_{60} = \text{EMA}_{60}(\text{BOP})$
- **物理含义**：力量平衡（Balance of Power）的60周期指数平滑，正值表示买方主导
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{BOP\_EMA}) \cdot |\text{BOP\_EMA}|^{1.5}$

**22. SP20_BOP_EMA_60M — BOP平滑^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{BOP\_EMA}) \cdot |\text{BOP\_EMA}|^{2.0}$

---

**23. SP15_Aroon_Osc_60M — Aroon振荡器^1.5**

- **原始因子**：$\text{Aroon\_Osc} = \dfrac{\text{AroonUp}_{60} - \text{AroonDown}_{60}}{100}$（缩放到 $[-1, 1]$）
- **物理含义**：衡量近期最高价/最低价出现时间的远近，正值表示近期创新高（上升趋势）
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{Aroon\_Osc}) \cdot |\text{Aroon\_Osc}|^{1.5}$

**24. SP20_Aroon_Osc_60M — Aroon振荡器^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{Aroon\_Osc}) \cdot |\text{Aroon\_Osc}|^{2.0}$

---

### 五、量价交互 (4个)

结合成交量信息的价格信号，捕捉量价背离与共振。

---

**25. SP15_Volume_ROC_20M — 成交量ROC^1.5**

- **原始因子**：$\text{Vol\_ROC}_{20} = \dfrac{V_t - V_{t-20}}{V_{t-20}}$
- **物理含义**：成交量20周期变化率，放量/缩量信号
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{Vol\_ROC}) \cdot |\text{Vol\_ROC}|^{1.5}$

**26. SP20_Volume_ROC_20M — 成交量ROC^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{Vol\_ROC}) \cdot |\text{Vol\_ROC}|^{2.0}$

---

**27. SP15_PriceVol_Delta_Prod — 价量变化乘积^1.5**

- **原始因子**：$\text{PV\_Prod} = \dfrac{(C_t - C_{t-1}) \cdot (V_t - V_{t-1})}{C_t \cdot V_t}$
- **物理含义**：价格变化与成交量变化的乘积（归一化），正值=价量同向（趋势确认），负值=价量背离
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{PV\_Prod}) \cdot |\text{PV\_Prod}|^{1.5}$

**28. SP20_PriceVol_Delta_Prod — 价量变化乘积^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{PV\_Prod}) \cdot |\text{PV\_Prod}|^{2.0}$

---

### 六、跨周期价差 (4个)

通过短周期与长周期信号的差异捕捉动量加速/衰竭信号。

---

**29. SP15_ROC_Diff_5_60 — ROC短长差^1.5**

- **原始因子**：$\text{ROC\_Diff} = \text{ROC}_5 - \text{ROC}_{60}$
- **物理含义**：短期动量与长期动量的差异，正值表示短期动量强于长期（动量加速）
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{ROC\_Diff}) \cdot |\text{ROC\_Diff}|^{1.5}$

**30. SP20_ROC_Diff_5_60 — ROC短长差^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{ROC\_Diff}) \cdot |\text{ROC\_Diff}|^{2.0}$

---

**31. SP15_RV_Ratio_5_60 — 波动率比^1.5**

- **原始因子**：$\text{RV\_Ratio} = \dfrac{\text{RV}_5}{\text{RV}_{60}} - 1$
  - $\text{RV}_5 = \sqrt{\sum_{i=0}^{4} r_{t-i}^2}$（短期实现波动率）
  - $\text{RV}_{60} = \sqrt{\sum_{i=0}^{59} r_{t-i}^2}$（长期实现波动率）
- **物理含义**：短期波动率与长期波动率的比值减1，正值=波动率放大（可能伴随趋势启动）
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{RV\_Ratio}) \cdot |\text{RV\_Ratio}|^{1.5}$

**32. SP20_RV_Ratio_5_60 — 波动率比^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{RV\_Ratio}) \cdot |\text{RV\_Ratio}|^{2.0}$

---

### 七、布林带/通道位置 (3个)

价格在波动率通道中的相对位置信号。

---

**33. SP15_BB_Position_60M — 布林带位置^1.5**

- **原始因子**：$\text{BB\_Pos} = \dfrac{C - \text{BB\_Lower}}{\text{BB\_Upper} - \text{BB\_Lower}} - 0.5$（中心化至 $[-0.5, 0.5]$）
  - $\text{BB\_Upper} = \text{SMA}_{20} + 2\sigma_{20}$，$\text{BB\_Lower} = \text{SMA}_{20} - 2\sigma_{20}$
- **物理含义**：收盘价在布林带内的相对位置（中心化），正值=接近上轨，负值=接近下轨
- **SignedPower变换**：$\text{SP15} = \text{sign}(\text{BB\_Pos}) \cdot |\text{BB\_Pos}|^{1.5}$

**34. SP20_BB_Position_60M — 布林带位置^2.0**

- **公式**：$\text{SP20} = \text{sign}(\text{BB\_Pos}) \cdot |\text{BB\_Pos}|^{2.0}$

---

**35. SP20_VWAP_Spread_60M — VWAP价差^2.0**

- **原始因子**：$\text{VWAP\_Spread} = \dfrac{C - \text{VWAP}_{60}}{C}$
  - $\text{VWAP}_{60} = \dfrac{\sum_{i=0}^{59} V_{t-i} \cdot \text{TP}_{t-i}}{\sum_{i=0}^{59} V_{t-i}}$，$\text{TP} = \frac{H+L+C}{3}$
- **物理含义**：收盘价与60周期成交量加权均价的相对偏离，机构交易者的参考成本线
- **SignedPower变换**：$\text{SP20} = \text{sign}(\text{VWAP\_Spread}) \cdot |\text{VWAP\_Spread}|^{2.0}$

---

### 第二轮设计思路总结

| 类别 | 因子数 | 设计动机 |
|------|:-----:|:--------|
| 多周期EMA乖离 (5M/15M) | 4 | 补全第一轮仅覆盖60M周期的不足 |
| 多周期MOM/ROC (5/15/60) | 6 | 构建完整动量周期谱系 |
| 趋势偏离 (HMA/KAMA/Ichimoku) | 6 | 引入自适应均线和云图信号 |
| 振荡器 (WILLR/CMO/BOP/Aroon) | 8 | 扩展TA-Lib振荡器覆盖 |
| 量价交互 (VolROC/PVProd) | 4 | 成交量维度信号 |
| 跨周期价差 (ROCDiff/RVRatio) | 4 | 捕捉多周期共振/背离 |
| 通道位置 (BB/VWAP) | 3 | 波动率通道内的位置信号 |

全部因子均应用 SP15 ($p=1.5$) 和 SP20 ($p=2.0$) 两种有符号幂变换进行优化对比。

# Signed Power 第二轮 — 拓展因子库覆盖

## 新增优化目标

根据第一轮优化结果，**有符号振荡型因子对 SP15/SP20 响应最好**。本轮拓展覆盖：

| 类别 | 新增因子数 | 说明 |
|------|:--:|------|
| **多周期EMA乖离** (5M/15M) | 4 | 类似ema_ret_60M的短周期版本 |
| **多周期MOM/ROC** (15M/30M) | 6 | 填补ROC_5M和ROC_60M之间的周期 |
| **趋势偏离类** (HMA/KAMA乖离、Ichimoku溢价) | 6 | 价格与各种均线的偏离度 |
| **振荡器类** (WILLR, CMO, Aroon, BOP) | 8 | TA-Lib振荡器中心化后优化 |
| **量价交互** (量价背离、成交量ROC) | 4 | 成交量变化率+量价乘积 |
| **多周期价差** (短-长ROC差, 短/长RV比) | 4 | 跨周期共振信号 |
| **布林带/Keltner位置** (%B中心化) | 3 | 通道位置信号 |

共 **35个新因子**，全部应用 **SP15** 和 **SP20** 两种幂次（因为正值压缩效果不佳）。

In [ ]:
# ==========================================
# Signed Power 第二轮 — 35个新优化因子
# ==========================================
import os, time, gc
import numpy as np
import pandas as pd
import talib
import bottleneck as bn
from pathlib import Path

BASE_DIR = Path(r"C:\Users\wangx\Desktop\华东")
RAW_DATA_DIR = BASE_DIR / "train_dataset" / "train_dataset"
OUTPUT_CSV3 = BASE_DIR / "SignedPower_Round2_IC_Evaluation.csv"
DATASET_COUNT = 30

SP2_FACTOR_NAMES = [
    # === 多周期EMA乖离 (4) ===
    "SP15_ema_ret_5M",            # EMA5乖离^1.5
    "SP20_ema_ret_5M",            # EMA5乖离^2.0
    "SP15_ema_ret_15M",           # EMA15乖离^1.5
    "SP20_ema_ret_15M",           # EMA15乖离^2.0
    # === 多周期MOM/ROC (6) ===
    "SP15_mom_5M",                # 5期动量^1.5
    "SP20_mom_5M",                # 5期动量^2.0
    "SP15_mom_60M",               # 60期动量^1.5
    "SP20_mom_60M",               # 60期动量^2.0
    "SP15_roc_15M",               # 15期ROC^1.5
    "SP20_roc_15M",               # 15期ROC^2.0
    # === 趋势偏离类 (6) ===
    "SP15_HMA_Deviation",         # (HMA-Close)/Close^1.5
    "SP20_HMA_Deviation",         # (HMA-Close)/Close^2.0
    "SP15_KAMA_Deviation",        # (KAMA-Close)/Close^1.5
    "SP20_KAMA_Deviation",        # (KAMA-Close)/Close^2.0
    "SP15_Ichimoku_Premium",      # Ichimoku溢价^1.5
    "SP20_Ichimoku_Premium",      # Ichimoku溢价^2.0
    # === 振荡器类 (8) ===
    "SP15_WILLR_60M",             # Williams %R^1.5
    "SP20_WILLR_60M",             # Williams %R^2.0
    "SP15_CMO_60M",               # Chande MO^1.5
    "SP20_CMO_60M",               # Chande MO^2.0
    "SP15_BOP_EMA_60M",           # BOP平滑^1.5
    "SP20_BOP_EMA_60M",           # BOP平滑^2.0
    "SP15_Aroon_Osc_60M",         # Aroon振荡^1.5
    "SP20_Aroon_Osc_60M",         # Aroon振荡^2.0
    # === 量价交互 (4) ===
    "SP15_Volume_ROC_20M",        # 成交量ROC^1.5
    "SP20_Volume_ROC_20M",        # 成交量ROC^2.0
    "SP15_PriceVol_Delta_Prod",   # dPrice*dVol^1.5
    "SP20_PriceVol_Delta_Prod",   # dPrice*dVol^2.0
    # === 多周期价差 (4) ===
    "SP15_ROC_Diff_5_60",         # ROC5-ROC60^1.5
    "SP20_ROC_Diff_5_60",         # ROC5-ROC60^2.0
    "SP15_RV_Ratio_5_60",         # (RV5/RV60-1)^1.5
    "SP20_RV_Ratio_5_60",         # (RV5/RV60-1)^2.0
    # === 布林带/通道位置 (3) ===
    "SP15_BB_Position_60M",       # %B中心化^1.5
    "SP20_BB_Position_60M",       # %B中心化^2.0
    "SP20_VWAP_Spread_60M",       # (Close-VWAP)/Close^2.0
]
FACTOR_DIM = len(SP2_FACTOR_NAMES)
print(f"第二轮: {FACTOR_DIM} 个Signed Power优化因子")

# ==========================================
# 拓展基础因子计算
# ==========================================
def compute_base_factors_round2(O, H, L, C, V):
    eps = 1e-8
    C_64 = C.astype(np.float64); H_64 = H.astype(np.float64)
    L_64 = L.astype(np.float64); V_64 = V.astype(np.float64)
    O_64 = O.astype(np.float64)
    T = len(C_64)

    # --- EMA乖离 多周期 ---
    ema5 = talib.EMA(C_64, timeperiod=5)
    ema5 = np.nan_to_num(ema5, nan=C_64[0])
    ema_ret_5 = ((C_64 - ema5) / (ema5 + eps)).astype(np.float32)
    ema15 = talib.EMA(C_64, timeperiod=15)
    ema15 = np.nan_to_num(ema15, nan=C_64[0])
    ema_ret_15 = ((C_64 - ema15) / (ema15 + eps)).astype(np.float32)

    # --- MOM (纯动量) ---
    mom_5 = (C_64 - np.roll(C_64, 5)).astype(np.float32); mom_5[:5] = 0.0
    mom_60 = (C_64 - np.roll(C_64, 60)).astype(np.float32); mom_60[:60] = 0.0

    # --- ROC 15M ---
    roc_15 = ((C_64 - np.roll(C_64, 15)) / (np.roll(C_64, 15) + eps)).astype(np.float32); roc_15[:15] = 0.0

    # --- HMA/KAMA Deviation ---
    p60 = 60
    kama60 = talib.KAMA(C_64, timeperiod=p60)
    kama60 = np.nan_to_num(kama60, nan=C_64[0])
    half_p = max(2, p60//2); sqrt_p = max(2, int(np.sqrt(p60)))
    wma_h = talib.WMA(C_64, timeperiod=half_p); wma_f = talib.WMA(C_64, timeperiod=p60)
    hma60 = talib.WMA(2*wma_h - wma_f, timeperiod=sqrt_p)
    hma60 = np.nan_to_num(hma60, nan=C_64[0])
    hma_dev = ((hma60 - C_64) / (C_64 + eps)).astype(np.float32)
    kama_dev = ((kama60 - C_64) / (C_64 + eps)).astype(np.float32)

    # --- Ichimoku Premium ---
    hh_30 = bn.move_max(H_64, window=30, min_count=1)
    ll_30 = bn.move_min(L_64, window=30, min_count=1)
    ichimoku_mid = (hh_30 + ll_30) / 2.0
    ichimoku_prem = ((C_64 - ichimoku_mid) / (ichimoku_mid + eps)).astype(np.float32)

    # --- WILLR ---
    willr_60 = talib.WILLR(H_64, L_64, C_64, timeperiod=p60)
    willr_60 = np.nan_to_num(willr_60, nan=-50.0) / 100.0  # scale to [-1,0]

    # --- CMO ---
    cmo_60 = talib.CMO(C_64, timeperiod=p60)
    cmo_60 = np.nan_to_num(cmo_60, nan=0.0) / 100.0  # scale to [-1,1]

    # --- BOP EMA ---
    bop = (C_64 - O_64) / (H_64 - L_64 + eps)
    bop_ema_60 = talib.EMA(bop, timeperiod=p60)
    bop_ema_60 = np.nan_to_num(bop_ema_60, nan=0.0)

    # --- Aroon Oscillator ---
    aroon_down, aroon_up = talib.AROON(H_64, L_64, timeperiod=p60)
    aroon_osc = np.nan_to_num(aroon_up - aroon_down, nan=0.0) / 100.0  # scale to [-1,1]

    # --- Volume ROC ---
    vol_roc_20 = ((V_64 - np.roll(V_64, 20)) / (np.roll(V_64, 20) + eps)).astype(np.float32); vol_roc_20[:20] = 0.0

    # --- Price-Volume Delta Product ---
    d_price = C_64 - np.roll(C_64, 1); d_price[0] = 0.0
    d_vol = V_64 - np.roll(V_64, 1); d_vol[0] = 0.0
    pv_delta_prod = (d_price * d_vol / (C_64 * V_64 + eps)).astype(np.float32)

    # --- ROC Diff & RV Ratio ---
    roc_5_full = (C_64 - np.roll(C_64, 5)) / (np.roll(C_64, 5) + eps); roc_5_full[:5] = 0.0
    roc_60_full = (C_64 - np.roll(C_64, 60)) / (np.roll(C_64, 60) + eps); roc_60_full[:60] = 0.0
    roc_diff_5_60 = (roc_5_full - roc_60_full).astype(np.float32)

    log_ret = np.log(np.maximum(C_64, eps)) - np.log(np.maximum(np.roll(C_64, 1), eps))
    log_ret[0] = 0.0
    rv5 = np.sqrt(bn.move_sum(log_ret**2, window=5, min_count=1))
    rv60_val = np.sqrt(bn.move_sum(log_ret**2, window=60, min_count=1))
    rv_ratio = ((rv5 / (rv60_val + eps)) - 1.0).astype(np.float32)

    # --- Bollinger Band Position ---
    sma20 = talib.SMA(C_64, timeperiod=20)
    sma20 = np.nan_to_num(sma20, nan=C_64[0])
    std20 = bn.move_std(C_64, window=20, min_count=1)
    bb_upper = sma20 + 2*std20; bb_lower = sma20 - 2*std20
    bb_pos = np.clip(((C_64 - bb_lower) / (bb_upper - bb_lower + eps) - 0.5), -0.5, 0.5).astype(np.float32)

    # --- VWAP Spread ---
    tp = (H_64 + L_64 + C_64) / 3.0
    vwap_60 = (bn.move_sum(V_64*tp, window=p60, min_count=1) / (bn.move_sum(V_64, window=p60, min_count=1) + eps))
    vwap_spread = ((C_64 - vwap_60) / (C_64 + eps)).astype(np.float32)

    return {
        'ema_ret_5': ema_ret_5, 'ema_ret_15': ema_ret_15,
        'mom_5': mom_5, 'mom_60': mom_60, 'roc_15': roc_15,
        'hma_dev': hma_dev, 'kama_dev': kama_dev,
        'ichimoku_prem': ichimoku_prem,
        'willr_60': willr_60, 'cmo_60': cmo_60,
        'bop_ema_60': bop_ema_60, 'aroon_osc': aroon_osc,
        'vol_roc_20': vol_roc_20, 'pv_delta_prod': pv_delta_prod,
        'roc_diff_5_60': roc_diff_5_60, 'rv_ratio': rv_ratio,
        'bb_pos': bb_pos, 'vwap_spread': vwap_spread,
    }


def signed_power(x, p):
    return np.sign(x) * (np.abs(x) ** p)


def generate_sp2_factors(O, H, L, C, V):
    bf = compute_base_factors_round2(O, H, L, C, V)
    f = []

    # ema_ret 5M/15M
    f.append(signed_power(bf['ema_ret_5'], 1.5))
    f.append(signed_power(bf['ema_ret_5'], 2.0))
    f.append(signed_power(bf['ema_ret_15'], 1.5))
    f.append(signed_power(bf['ema_ret_15'], 2.0))

    # mom/roc
    f.append(signed_power(bf['mom_5'], 1.5))
    f.append(signed_power(bf['mom_5'], 2.0))
    f.append(signed_power(bf['mom_60'], 1.5))
    f.append(signed_power(bf['mom_60'], 2.0))
    f.append(signed_power(bf['roc_15'], 1.5))
    f.append(signed_power(bf['roc_15'], 2.0))

    # trend deviation
    f.append(signed_power(bf['hma_dev'], 1.5))
    f.append(signed_power(bf['hma_dev'], 2.0))
    f.append(signed_power(bf['kama_dev'], 1.5))
    f.append(signed_power(bf['kama_dev'], 2.0))
    f.append(signed_power(bf['ichimoku_prem'], 1.5))
    f.append(signed_power(bf['ichimoku_prem'], 2.0))

    # oscillators
    f.append(signed_power(bf['willr_60'], 1.5))
    f.append(signed_power(bf['willr_60'], 2.0))
    f.append(signed_power(bf['cmo_60'], 1.5))
    f.append(signed_power(bf['cmo_60'], 2.0))
    f.append(signed_power(bf['bop_ema_60'], 1.5))
    f.append(signed_power(bf['bop_ema_60'], 2.0))
    f.append(signed_power(bf['aroon_osc'], 1.5))
    f.append(signed_power(bf['aroon_osc'], 2.0))

    # volume-price interaction
    f.append(signed_power(bf['vol_roc_20'], 1.5))
    f.append(signed_power(bf['vol_roc_20'], 2.0))
    f.append(signed_power(bf['pv_delta_prod'], 1.5))
    f.append(signed_power(bf['pv_delta_prod'], 2.0))

    # multi-timeframe spreads
    f.append(signed_power(bf['roc_diff_5_60'], 1.5))
    f.append(signed_power(bf['roc_diff_5_60'], 2.0))
    f.append(signed_power(bf['rv_ratio'], 1.5))
    f.append(signed_power(bf['rv_ratio'], 2.0))

    # channel/band positions
    f.append(signed_power(bf['bb_pos'], 1.5))
    f.append(signed_power(bf['bb_pos'], 2.0))
    f.append(signed_power(bf['vwap_spread'], 2.0))

    return np.column_stack(f).astype(np.float32)


def calculate_pearson_ic(factors, label):
    vm = ~np.isnan(label); y = label[vm]; X = factors[vm, :]
    nm = np.isnan(X); X = np.where(nm,0,X)
    vc = (~nm).sum(axis=0); vc = np.where(vc==0,1e-8,vc)
    ym=np.mean(y); xm=X.sum(axis=0)/vc
    yc=y-ym; xc=np.where(nm,0,X-xm)
    cov=np.dot(yc,xc)/vc
    ys=np.sqrt(np.mean(yc**2)); xs=np.sqrt(np.sum(xc**2,axis=0)/vc)
    ic=cov/(ys*xs+1e-8); ic=np.where(vc<len(y)*0.1,0,ic)
    return ic


def evaluate_sp2():
    print("="*70)
    print("[Signed Power 第二轮] 35个新优化因子 IC评估")
    print("="*70)
    ic5=np.zeros((DATASET_COUNT,FACTOR_DIM),dtype=np.float32)
    ic60=np.zeros((DATASET_COUNT,FACTOR_DIM),dtype=np.float32)
    success=0; t0=time.time()

    for i in range(DATASET_COUNT):
        rp=RAW_DATA_DIR/f"dataset{i}_train_ohlcv.npy"
        if not rp.exists(): continue
        try:
            raw=np.load(rp).astype(np.float32)
            if raw.shape[1]!=8: continue
            O,H,L,C,V=raw[:,1],raw[:,2],raw[:,3],raw[:,4],raw[:,5]
            spf=generate_sp2_factors(O,H,L,C,V)
            ic5[i,:]=calculate_pearson_ic(spf,raw[:,6])
            ic60[i,:]=calculate_pearson_ic(spf,raw[:,7])
            success+=1
            if success%5==0: print(f"  已完成 {success}/{DATASET_COUNT} (耗时{(time.time()-t0)/60:.1f}分)...")
            del raw,spf; gc.collect()
        except Exception as e: print(f"  dataset{i} 失败: {e}")

    print(f"\n[完成] {success}/{DATASET_COUNT}, 耗时 {(time.time()-t0)/60:.2f} 分钟")
    vic5=ic5[:success]; vic60=ic60[:success]
    m5=np.nanmean(vic5,0); m60=np.nanmean(vic60,0)
    s5=np.nanstd(vic5,0); s60=np.nanstd(vic60,0)
    ir5=m5/(s5+1e-8); ir60=m60/(s60+1e-8)
    sc5=np.abs(m5)*np.abs(ir5); sc60=np.abs(m60)*np.abs(ir60)

    df=pd.DataFrame({
        'Factor_Name':SP2_FACTOR_NAMES,
        'Ret5_Mean_IC':m5,'Ret5_Std_IC':s5,'Ret5_IC_IR':ir5,
        'Ret60_Mean_IC':m60,'Ret60_Std_IC':s60,'Ret60_IC_IR':ir60,
        'Ret5_Score':sc5,'Ret60_Score':sc60,
        'Comprehensive_Score':sc5+sc60
    })

    print("\n[Ret5 Top 10]")
    t5=df.sort_values('Ret5_Score',ascending=False).head(10)
    print(t5[['Factor_Name','Ret5_Mean_IC','Ret5_IC_IR','Ret5_Score']].to_string(index=False))

    print("\n[Ret60 Top 10]")
    t60=df.sort_values('Ret60_Score',ascending=False).head(10)
    print(t60[['Factor_Name','Ret60_Mean_IC','Ret60_IC_IR','Ret60_Score']].to_string(index=False))

    print("\n[综合排名]")
    dfs=df.sort_values('Comprehensive_Score',ascending=False)
    print(dfs[['Factor_Name','Ret5_Mean_IC','Ret60_Mean_IC','Comprehensive_Score']].to_string(index=False))

    # 合并两轮Top因子
    print("\n" + "="*70)
    print("[两轮汇总 Top 20 — Signed Power全部优化因子]")
    print("="*70)
    # Read round 1 results
    df_r1 = pd.read_csv(BASE_DIR / "SignedPower_Optimized_IC_Evaluation.csv")
    df_all = pd.concat([df_r1, df], ignore_index=True)
    df_all_sorted = df_all.sort_values('Comprehensive_Score', ascending=False).head(20)
    print(df_all_sorted[['Factor_Name','Ret5_Mean_IC','Ret5_IC_IR','Ret60_Mean_IC','Ret60_IC_IR','Comprehensive_Score']].to_string(index=False))

    df.to_csv(OUTPUT_CSV3, index=False, encoding='utf-8-sig')
    print(f"\n[保存] {OUTPUT_CSV3}")
    return df

if __name__ == "__main__":
    df_sp2 = evaluate_sp2()
